# Controller pipeline

Calibrate -> noise -> record -> view -> fly. Run setup, then any stage on its own.
Each stage's detail is in its own `theory.md`; this is only the driver.

In [1]:
from controller.camera import identify

for d in identify.connected():
    print(f"{d.name:32s} {d.unique_id}")
print("ELP indices:", identify.elp_indices())
# Printed, deliberately not saved: the device list moves, so a captured variable goes
# stale. capture() and record() resolve it themselves at the moment they open.

# ---------------------------------------------------------------- camera mode
# ONE source of truth for every stage below. Measured on this rig 2026-08-29
# (ELP OV9281, mono global shutter). Only these two are true modes -- 480x300 and
# 320x200 silently fall back to 640x400.
#
#   name      W x H      fps   rotor alias limit   exposure   note
#   native  1280 x 800   119        14.9 Hz         8.4 ms    full field of view
#   fast     640 x 400   210        26.2 Hz         4.8 ms    exact 0.5x, same FOV
#   probe    320 x 240   364        45.5 Hz         2.8 ms    CROPPED, see below
#
# "probe" is NOT a rescale, it is a sensor crop: the rotor overflows the frame, the
# ring fit lands on a 367 px ellipse in a 320 px image, and blade_phase strength
# collapses from 0.545 to 0.033. Contrast is fine (RMS 0.784 vs 0.788, Michelson
# 0.960 vs 0.954) and it is brighter, not darker -- the problem is framing alone.
# Usable only if the camera is physically moved back so the rotor fits.
MODES = {"native": (1280, 800), "fast": (640, 400), "probe": (320, 240)}
# "fast" since 2026-08-30: a true 0.5x of native, so the rig rescales exactly and the
# pipeline runs 20.8 -> 64.8 Hz for the 0.119 mm pose/theory.md 13 already priced.
# "probe" was measured and rejected -- 6 Hz more for twice the bias. See theory.md 19.3.
MODE = "fast"
WIDTH, HEIGHT = MODES[MODE]
print(f"mode {MODE}: {WIDTH}x{HEIGHT}")

Global Shutter Camera            0x12000032e49281
Global Shutter Camera            0x11000032e49281
FaceTime HD Camera               47B4B64B-7067-4B9C-AD2B-AE273A71F4B5
Kevin’s iPhone Camera            FC022C17-ACC0-48CA-A8B2-38D100000001
ELP indices: [0, 1]
mode fast: 640x400


## 1. Calibrate

Board -> intrinsics -> extrinsics -> `calib/stereo_rig.json`, which also records
*which* two cameras these were, so later stages can find them again.

In [2]:
from controller.calib.calibrate import PAIR_DIR, SPEC, run_calibration
from controller.calib.capture import capture
from controller.calib.plots import undistort_figure
from controller.calib.results import write_results
from controller.calib.rig import check_mode

# Shoots at the mode selected in the setup cell, so the intrinsics that come out
# belong to the mode you will actually fly. 640x400 costs absolute scale -- each pixel
# is worth twice the mm, so position noise in mm doubles -- but the fit is just as
# good, rms/major is 0.0022 at both, measured.
capture(PAIR_DIR, spec=SPEC, width=WIDTH, height=HEIGHT, override=(MODE != "native"))

cal = run_calibration(SPEC, PAIR_DIR)
write_results(cal, SPEC) if cal["passed"] else print("gate failed, nothing written")
undistort_figure(cal)                              # straight edges should straighten
check_mode(WIDTH, HEIGHT)

/Users/meli/Desktop/Kevin/UCB/FlyingRobotsActiveControl/ESP32_PMW/results/stereo_calibration/pairs already holds 176 image(s); override: they will be replaced
auto: SPACE pauses and resumes, q quits


Image(value=b'', format='jpeg')


interrupted

0 pair(s) in /Users/meli/Desktop/Kevin/UCB/FlyingRobotsActiveControl/ESP32_PMW/results/stereo_calibration/pairs, plus 0 solo for A, 0 solo for B
camera A: 94 usable views, 31.6 corners mean
camera B: 82 usable views, 29.9 corners mean
image size (1280, 800)

camera A: 91 views, RMS 0.2948 px, worst view 0.6976 px, 3 over 60 deg)
  fx=2753.63 fy=2750.42 cx=672.89 cy=386.14
  +/-  1.75      1.98      3.52     2.16   (1 sigma)
  dist [-3.08982e-01 -7.90162e-01 -1.91040e-03  1.15198e-03  1.16399e+01]
  incidence 11.5-59.8 deg, orientation spread 24.4 deg

camera B: 78 views, RMS 0.3400 px, worst view 0.7188 px, 4 over 60 deg)
  fx=2659.10 fy=2661.76 cx=518.91 cy=354.33
  +/-  2.11      2.52      4.73     2.61   (1 sigma)
  dist [-3.30220e-01  4.04464e-01  3.68788e-04  1.89014e-03 -3.20483e+00]
  incidence 15.9-59.3 deg, orientation spread 22.2 deg

15 usable pairs, 3 rejected
  reject pair_014: incidence 21/62 deg
  reject pair_015: incidence 61/29 deg
  reject pair_016: inci

True

In [ ]:
# Re-fit from the images already in PAIR_DIR, without re-shooting them.
cal = run_calibration(SPEC, PAIR_DIR)
write_results(cal, SPEC) if cal["passed"] else print("gate failed, nothing written")
undistort_figure(cal)
check_mode(WIDTH, HEIGHT)

## 2. Static noise

How much the pose scatters when the robot is *not* moving -- that scatter is the
measurement noise. Artifact is `pose/noise_model.json`; everything downstream loads
it on its own.

In [ ]:
from controller.pose import noise

# print(noise.NoiseModel.load().summary())     # what is loaded right now

# Re-record. Clamp the robot and leave the coils off: this measures the scatter around
# a truth that is not moving, so anything that really moves it is measured as noise.
# Returns at once and puts a preview + buttons below: aim until BOTH views lock an
# ellipse, press "Shoot station", move the robot to a new height, repeat. Three or four
# heights, because sigma_depth = frac * z and one height cannot separate frac from a
# constant. It writes noise_model.json on the last station.

# session = noise.record_live(stations=4)
# print(session.summary())                   # the report, once it says done

## 3. Record

Stereo mp4 per flight, each take in its own dated folder.

In [3]:
from controller.camera.record import DEFAULT_DIR, latest_flight, record

# Preview renders below the cell. Starting/stopping a take needs SPACE, which a cell
# has no way to send, so shoot flights from a terminal:
#   uv run python controller/camera/record.py
record(DEFAULT_DIR)                        # preview only from here; interrupt to stop

Image(value=b'', format='jpeg')

source ended

0 flight(s) in /Users/meli/Desktop/Kevin/UCB/FlyingRobotsActiveControl/ESP32_PMW/results/flights
  capture skew: {'n': 84056, 'median_ms': 0.9791669726837426, 'p95_ms': 8.57336500484962, 'max_ms': 84.33241600869223, 'dropped': 0}


[]

## 4. Visualise

5-DOF pose estimation in viser. The datum puts the rotor axis on +z and the scene
follows it; the datum line printed at startup says where up ended up.

In [ ]:
# from controller.viz.live_viz import from_recording, from_stereo, replay
# from controller.calib.rig import StereoRig

# from_stereo(StereoRig.load().sources())      # live; interrupt to stop

# from controller.pose import background; background.capture_stereo()
#                                   # optional saved plate, robot out of frame;
#                                   # the default builds one from the stream
# flight = latest_flight(DEFAULT_DIR)
# from_recording(flight, csv_out=flight / "poses.csv")   # offline, from mp4
# replay(flight / "poses.csv")                           # offline, from the CSV

## 5. Fly

Closed loop off the camera. Rehearse dry first -- the live line energises coils.

In [ ]:
from controller.control import hover_controller_runner as r
from controller.control import ramp
from controller.calib.rig import StereoRig, check_mode

pair = StereoRig.load().sources()      # A,B in calibrated order, whatever the indices
check_mode(WIDTH, HEIGHT)              # intrinsics must match MODE, or every mm is wrong

# Open loop first: takeoff=False so the ramp waits for the button, enable_freq_cmd=False
# so the altitude loop never touches the frequency. The point of this run is to measure
# where the robot leaves the pad, and a loop trimming f while it happens erases that.
r.fly(source="camera", camera=pair, width=WIDTH, height=HEIGHT,
      takeoff=False, enable_freq_cmd=False, record=True, log="hover_run.log")

#   The ramp: `controller/control/ramp.py` owns the shape, `constants.py` owns the
#   numbers. `ramp.DEFAULT` is what flew on 2026-08-29. To try another, pass
#   `segments=` -- `ramp.check` refuses gaps and over-cap totals rather than clamping:
#     r.fly(..., segments=((2, 6, 8.0, ramp.EASE, 2), (6, 210, 22.0, ramp.EASE, 2)))
#
#   Coil heat: fly() BLOCKS until the coils can take a run this long. There is no
#   temperature sensor -- ai/thermal/coil_thermal.py models it, and fly() refuses to
#   arm if that file is missing. Coils reached 80 C after four ramps.
#
#   Control panel, in order. Both keys start safe:
#     stop            coils off now, and clears armed / mag max.
#     armed           OFF -- nothing reaches the coils.
#     x / y / z       setpoint in mm, datum frame.
#     gain lateral    scales the lateral loop; gain vertical scales the altitude loop.
#     mag max         0.00 -- clamps lateral even when armed.
#     land / takeoff  takeoff runs the spin-up ramp on demand; land ramps down.
#   So: takeoff -> watch z and f -> tick armed -> raise mag max slowly.
#
#   NOTHING STOPS THE COILS AUTOMATICALLY. No firmware watchdog, no host watchdog, no
#   timeout. The viser stop button and the GPIO14 button on the board are the only kills.
#
#   Iterate: run -> read the printed metrics block (rotor / f_liftoff / verdict) ->
#     from controller.control import takeoff_report; takeoff_report.compare(5)
#   to overlay the last five attempts, then change one ramp knob and go again.
#   Did the rotor turn at all?  uv run python ai/spinup/detector.py
#   Always finish with: uv run python controller/control/safe_off.py